# 2. Simple usage

Five things you can do immediately, each one call.

1. **`explain`** - what does this function do at this point?
2. **derivatives** - every order from one evaluation
3. **`0/0`** - resolved without L'Hopital
4. **`audit`** - will my formula lose digits, before I know the answer?
5. **`TAG` / `to_ieee754`** - classical derivatives, and the way back to floats

In [1]:
import math, warnings
import composite.composite_lib as cl
from composite import (R, ZERO, Composite, sin, cos, exp, ln, sqrt,
                       explain, audit, forensics, TAG, UNTAG)
from composite.backends.config import use_dict

use_dict(); cl._refresh_constants()
# MAX_ACTIVE_DIMS is left at its default of 60. Raising it removes the
# truncation guard, and an iterative algorithm then grows terms without
# bound -- solve_ode below did not finish 20 steps in 30s at 10**9, and
# finishes 200 steps in 0.86s at the default.
warnings.simplefilter("ignore")
for w in (cl.CancellationWarning, cl.NotConventionalWarning):
    warnings.filterwarnings("ignore", category=w)
print("ready")

ready


## 1. What does this function do at this point?

Float arithmetic answers `nan` wherever the question is interesting. `explain`
seeds the point with an infinitesimal and reads the answer off the grades.

In [2]:
questions = [("sin(x)/x        at 0", lambda x: sin(x) / x,            0.0),
             ("1/x             at 0", lambda x: R(1) / x,              0.0),
             ("1/(x-2)**2      at 2", lambda x: R(1) / (x - R(2))**2,  2.0),
             ("sqrt(x)         at 0", lambda x: sqrt(x),               0.0),
             ("ln(x)           at 0", lambda x: ln(x),                 0.0),
             ("x*ln(x)         at 0", lambda x: x * ln(x),             0.0),
             ("x*x + 3x        at 2", lambda x: x * x + R(3) * x,      2.0)]
for label, f, at in questions:
    print("%-22s %s" % (label, explain(f, at)))

sin(x)/x        at 0   f at x = 0: value 1; slope 0; numerically stable
1/x             at 0   f at x = 0: blows up like +1 / x
1/(x-2)**2      at 2   f at x = 2: blows up like +1 / (x-2)**2
sqrt(x)         at 0   f at x = 0: value 0, but with a corner: it leaves the point like x**0.5, so the slope is infinite
ln(x)           at 0   f at x = 0: grows without bound as x approaches 0, but only logarithmically (slower than any power of 1/x)
x*ln(x)         at 0   f at x = 0: value 0, but the slope is unbounded (it diverges logarithmically); numerically stable
x*x + 3x        at 2   f at x = 2: value 10; slope 7; numerically stable


`kind` is machine-readable, so this is usable in code and not only for reading.

In [3]:
e = explain(lambda x: R(1) / (x - R(2))**2, 2.0)
print("kind       ", e.kind)
print("order      ", e.order)
print("coefficient", e.coefficient)
print("stability  ", e.stability)
e

kind        unbounded
order       2.0
coefficient 1.0
stability   None


field,value
kind,unbounded
order,2
coefficient,1
degeneracy,"one infinitesimal entered, so grade -n is the nth derivative in the ordinary sense"


## 2. Every derivative, from one evaluation

Seed the point, evaluate the formula once, read the grades. Checked here against
40-digit arithmetic.

In [4]:
from mpmath import mp, mpf
mp.dps = 40

f_composite = lambda x: sin(x) * exp(-(x * x)) / (R(1) + x * x)
f_exact     = lambda t: mp.sin(t) * mp.e ** (-(t * t)) / (1 + t * t)
at = 0.7

jet = f_composite(R(at) + ZERO)
print("%-6s %-24s %-24s %s" % ("order", "from the composite", "40-digit truth", "rel err"))
for k in range(5):
    got  = jet.d(k)
    want = float(mp.diff(f_exact, mpf(at), k))
    print("%-6d %-24.15g %-24.15g %.1e" % (k, got, want, abs(got - want) / max(abs(want), 1)))

order  from the composite       40-digit truth           rel err
0      0.26487567704849         0.26487567704849         0.0e+00
1      -0.305230938842161       -0.305230938842161       5.6e-17
2      -0.937940339717236       -0.937940339717236       1.1e-16
3      6.1313831610377          6.1313831610377          1.4e-16
4      -8.31997509321325        -8.31997509321326        6.4e-16


## 3. `0/0`, without L'Hopital

A removable singularity is not a special case here. The numerator and denominator
both vanish, their grades cancel, and grade 0 is the limit.

In [5]:
for label, f, at, want in [
        ("(x**3 - 8)/(x - 2)   at 2", lambda x: (x**3 - R(8)) / (x - R(2)), 2.0, 12.0),
        ("sin(x)/x             at 0", lambda x: sin(x) / x,                 0.0, 1.0),
        ("(1 - cos x)/x**2     at 0", lambda x: (R(1) - cos(x)) / (x * x),  0.0, 0.5),
        ("(exp(x) - 1)/x       at 0", lambda x: (exp(x) - R(1)) / x,        0.0, 1.0),
        ("ln(1 + x)/x          at 0", lambda x: ln(R(1) + x) / x,           0.0, 1.0)]:
    y = f(cl._seeded(at))
    print("%-28s = %-12.12g want %-6g  and the slope comes too: %.12g"
          % (label, y.st(), want, y.d(1)))

(x**3 - 8)/(x - 2)   at 2    = 12           want 12      and the slope comes too: 6
sin(x)/x             at 0    = 1            want 1       and the slope comes too: 0
(1 - cos x)/x**2     at 0    = 0.5          want 0.5     and the slope comes too: 0
(exp(x) - 1)/x       at 0    = 1            want 1       and the slope comes too: 0.5
ln(1 + x)/x          at 0    = 1            want 1       and the slope comes too: -0.5


## 4. Will this formula lose digits?

`audit` predicts the relative error from one evaluation, with no reference answer,
and separates **a bad formula** from **a hard problem**.

* `floor` - the best any formula could do on this problem
* `predicted` - what this formula is expected to stay under
* `predicted / floor` large means *rewrite it*; near 1 means *nothing to fix*

In [6]:
F = forensics.F
pairs = [("(1 - cos x)/x**2     @1e-5", lambda t: (R(1) - F.cos(t)) / (t * t), 1e-5),
         ("2 sin(x/2)**2/x**2   @1e-5", lambda t: R(2) * F.sin(t / R(2))**2 / (t * t), 1e-5),
         ("sqrt(x+1) - sqrt(x)  @1e8",  lambda t: F.sqrt(t + R(1)) - F.sqrt(t), 1e8),
         ("1/(sqrt(x+1)+sqrt x) @1e8",  lambda t: R(1) / (F.sqrt(t + R(1)) + F.sqrt(t)), 1e8)]
print("%-28s %-10s %-10s %-10s %s" % ("formula", "floor", "predicted", "pred/floor", "verdict"))
for label, f, at in pairs:
    a = audit(f, at)
    print("%-28s %-10.1e %-10.1e %-10.1e %s"
          % (label, a.floor, a.predicted, a.predicted / a.floor,
             getattr(a.verdict, "name", a.verdict)))

formula                      floor      predicted  pred/floor verdict
(1 - cos x)/x**2     @1e-5   2.2e-16    4.4e-06    2.0e+10    unstable formula
2 sin(x/2)**2/x**2   @1e-5   2.2e-16    1.8e-15    8.0e+00    stable
sqrt(x+1) - sqrt(x)  @1e8    2.2e-16    1.1e-07    5.0e+08    unstable formula
1/(sqrt(x+1)+sqrt x) @1e8    2.2e-16    7.2e-16    3.3e+00    stable


Each pair is the same mathematics spelled two ways. The verdict flips, and the
stable spelling is the one to ship.

## 5. Classical derivatives on demand

By default a written zero is an infinitesimal, so `x**2 + 0` has derivative 7 at 3,
not 6 - it is the exact derivative of what the expression *denotes*. When you want
the textbook answer instead, `TAG` the one value that should carry the infinitesimal
and every other zero goes conventional.

In [7]:
expr = lambda x: x * x + R(0)          # one written zero, nothing else

UNTAG()
provenance = expr(R(3) + ZERO)
x = TAG(3)
classical = expr(x)
UNTAG()

print("provenance mode (default) :", [provenance.d(k) for k in range(3)])
print("after TAG(3)              :", [classical.d(k) for k in range(3)])
print("the classical jet of x**2 :", [9.0, 6.0, 2.0])
print()
print("the 7 is not an error: it is the exact derivative of x**2 + (x-3),")
print("which is what a written zero at grade 0 denotes.")

provenance mode (default) : [9.0, 7.0, 2.0]
after TAG(3)              : [9.0, 6.0, 2.0]
the classical jet of x**2 : [9.0, 6.0, 2.0]

the 7 is not an error: it is the exact derivative of x**2 + (x-3),
which is what a written zero at grade 0 denotes.


## And the way back to floats

`to_ieee754` is the one well-defined projection into ordinary float arithmetic:
infinitesimals become true zeros, poles become signed infinities, NOTHING becomes
`nan`.

In [8]:
for label, v in (("x*x at 3",        (R(3) + ZERO) * (R(3) + ZERO)),
                ("R(6) - R(6)",      R(6) - R(6)),
                ("1/ZERO",           R(1) / ZERO),
                ("-1/ZERO",          R(-1) / ZERO),
                ("NOTHING",          Composite({}))):
    print("%-16s %-24s ->  %s" % (label, v, v.to_ieee754()))

x*x at 3         <|9|₀ |6|₋₁ |1|₋₂>       ->  9.0
R(6) - R(6)      <|6|₋₁>                  ->  0.0
1/ZERO           <|1|₁>                   ->  inf
-1/ZERO          <|-1|₁>                  ->  -inf
NOTHING          ∅                        ->  nan


## Summary

| want | call |
|---|---|
| what happens at this point | `explain(f, at)` |
| every derivative | seed with `R(a) + ZERO`, read `.d(k)` |
| a `0/0` limit | just evaluate it |
| accuracy of a formula | `audit(f, at)` and compare `predicted` to `floor` |
| the classical jet | `TAG(a)` ... `UNTAG()` |
| a plain float | `.to_ieee754()` |

Next: **3. Advanced concepts** - branch points, the log axis, divergent series,
and where this beats the alternatives.